# 220210_F1_run6: Decompositions

Stage 1 for the restarted V2a workflow. This notebook fits each configured linear BSS method and saves only decomposition artifacts: components, spectra, mixing matrices, means, run metadata, a summary table, and lightweight preview figures.

Default component policy: FastICA and Infomax fit the full trace count (`min(n_neurons, n_frames)`). SOBI and JADE use a PCA-reduced rank chosen as the smallest number of PCs preserving at least 95% variance, unless `PCA_COMPONENTS` is set explicitly. The selected rank and achieved variance are saved in metadata and in `decomposition_summary.csv`.

Run this before `cluster_clean_reconstruct.ipynb`. Do not select components here; cluster selection and cleaned-trace reconstruction happen in the next notebook. Leakage-safe behavior evidence is produced separately by `behavior_decoding.ipynb`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists() and (path / "src").exists()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

pd.set_option("display.max_colwidth", 120)
print(f"Project root: {PROJECT_ROOT}")


from ica_denoising.bss_notebook import (
    BSS_METHODS,
    available_datasets,
    load_traces,
    output_directory,
    resolve_bss_component_selection,
    run_bss_decomposition,
    summarize_decomposition_results,
)

In [ ]:
DATASET_KEY = "v2a-RSNs/220210_F1_run6_fluorescence"
METHODS_TO_RUN = list(BSS_METHODS) # ["fastica", "infomax"]
N_COMPONENTS = None  # None defaults to the filtered trace count for every BSS method.
PCA_COMPONENTS = None  # None defaults to the filtered trace count for every BSS method.
SOBI_JADE_PCA_VARIANCE_THRESHOLD = 0.95
TOLERANCE = 0.0001
MAX_ITERATIONS = 500
RANDOM_STATE = 0
SAVE_DECOMPOSITION_OUTPUTS = True

available_datasets(PROJECT_ROOT).query("group == 'v2a-RSNs'")


In [ ]:
dataset, traces = load_traces(DATASET_KEY, PROJECT_ROOT)
DEFAULT_COMPONENT_COUNT = traces.shape[0]
resolved_n_components = DEFAULT_COMPONENT_COUNT if N_COMPONENTS is None else N_COMPONENTS
resolved_pca_components = DEFAULT_COMPONENT_COUNT if PCA_COMPONENTS is None else PCA_COMPONENTS
OUTPUT_DATA_NAME = dataset.recording_id or dataset.data_name
component_plan = pd.DataFrame(
    [
        {
            "method": method,
            "n_components": selection.n_components,
            "pca_components": selection.pca_components,
            "pca_variance_threshold": selection.pca_variance_threshold,
            "pca_explained_variance_ratio": selection.pca_explained_variance_ratio,
            "component_selection_mode": selection.component_selection_mode,
        }
        for method in METHODS_TO_RUN
        for selection in [
            resolve_bss_component_selection(
                traces,
                method,
                n_components=resolved_n_components,
                pca_components=resolved_pca_components,
                pca_variance_threshold=SOBI_JADE_PCA_VARIANCE_THRESHOLD,
            )
        ]
    ]
)
output_dirs = {
    method: output_directory(
        method,
        OUTPUT_DATA_NAME,
        PROJECT_ROOT,
        analysis_kind="linear",
        dataset_group=dataset.group,
    )
    for method in METHODS_TO_RUN
}
dataset_output_dir = next(iter(output_dirs.values())).parent
figure_dir = dataset_output_dir / "figures"
if SAVE_DECOMPOSITION_OUTPUTS:
    figure_dir.mkdir(parents=True, exist_ok=True)

print(f"Dataset: {dataset.key}")
print(f"Trace file: {dataset.trace_path.relative_to(PROJECT_ROOT)}")
print(f"Trace shape (neurons x frames): {traces.shape}")
print(f"Sample rate: {dataset.sample_rate_hz:g} Hz")
display(component_plan)
pd.DataFrame({"method": output_dirs.keys(), "output_dir": output_dirs.values()})


In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
time_slice = slice(0, min(500, traces.shape[1]))
shown = range(min(8, traces.shape[0]))
offset = float(np.nanstd(traces[:, time_slice])) or 1.0
for row, neuron in enumerate(shown):
    ax.plot(traces[neuron, time_slice] + row * offset, lw=0.8)
ax.set(title="Raw fluorescence traces", xlabel="Frame", yticks=[])
fig.tight_layout()
raw_trace_plot_path = figure_dir / f"raw_trace_preview_{dataset.data_name}.png"
if SAVE_DECOMPOSITION_OUTPUTS:
    fig.savefig(raw_trace_plot_path, dpi=300, bbox_inches="tight")
    print(f"Saved raw trace preview: {raw_trace_plot_path.relative_to(PROJECT_ROOT)}")


In [ ]:
results = {}
for method in METHODS_TO_RUN:
    plan_row = component_plan[component_plan["method"] == method].iloc[0]
    print(
        f"Running {method} with n_components={plan_row['n_components']}, "
        f"pca_components={plan_row['pca_components']}, "
        f"mode={plan_row['component_selection_mode']}"
    )
    results[method] = run_bss_decomposition(
        dataset_key=DATASET_KEY,
        method=method,
        n_components=resolved_n_components,
        pca_components=resolved_pca_components,
        pca_variance_threshold=SOBI_JADE_PCA_VARIANCE_THRESHOLD,
        save_outputs=SAVE_DECOMPOSITION_OUTPUTS,
        project_root=PROJECT_ROOT,
        tol=TOLERANCE,
        max_iter=MAX_ITERATIONS,
        random_state=RANDOM_STATE,
        analysis_kind="linear",
    )

summary = summarize_decomposition_results(results)
summary_path = dataset_output_dir / "decomposition_summary.csv"
if SAVE_DECOMPOSITION_OUTPUTS:
    summary.to_csv(summary_path, index=False)
    print(f"Saved decomposition summary: {summary_path.relative_to(PROJECT_ROOT)}")
summary


In [ ]:
fig, axes = plt.subplots(len(results), 1, figsize=(14, 2.4 * len(results)), sharex=True)
axes = np.atleast_1d(axes)
for ax, (method, result) in zip(axes, results.items()):
    for component in range(min(8, result.ic_comps.shape[1])):
        ax.plot(result.ic_comps[:500, component] + component * 4, lw=0.7)
    ax.set(title=f"{method}: first independent components", yticks=[])
axes[-1].set_xlabel("Frame")
fig.tight_layout()
ic_preview_path = figure_dir / f"decomposition_ic_preview_{dataset.data_name}.png"
if SAVE_DECOMPOSITION_OUTPUTS:
    fig.savefig(ic_preview_path, dpi=300, bbox_inches="tight")
    print(f"Saved IC preview: {ic_preview_path.relative_to(PROJECT_ROOT)}")


In [ ]:
for method, result in results.items():
    print(f"{method}: {result.output_dir.relative_to(PROJECT_ROOT)}")
    for label, artifact_path in result.saved_paths.items():
        print(f"  {label}: {artifact_path.relative_to(PROJECT_ROOT)}")
if SAVE_DECOMPOSITION_OUTPUTS:
    print(f"summary: {summary_path.relative_to(PROJECT_ROOT)}")
    print(f"figures: {figure_dir.relative_to(PROJECT_ROOT)}")
